# 19 — Strong-model zero-shot sanity check

**Question:** does the selection–sufficiency failure remain in a substantially stronger instruction-tuned model, or is the current benchmark close to ceiling once model capacity is large enough?

This is intentionally minimal: **no training**, one 7B model, one seed, 50 examples per condition for each of the three entity-disjoint split definitions. The benchmark construction and scoring match the Experiment 15–18 family.

Primary readout: Distractor-5 (selection) vs Hard no-path (sufficiency). If both are near ceiling, the current paper should not frame the phenomenon as a general LLM limitation. If D5 remains high while Hard no-path remains materially lower, the dissociation survives a stronger-model sanity check.


In [ ]:
!pip -q install -U transformers accelerate bitsandbytes sentencepiece requests pandas

import os,re,gc,json,random,gzip,time
from pathlib import Path
import numpy as np, pandas as pd, torch, requests
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig, set_seed

MODEL_NAME='Qwen/Qwen2.5-7B-Instruct'
SEED=1
SPLITS=['ChemicalID','GeneID','DiseaseID']
N_EVAL=50
BATCH_SIZE=4
MAX_NEW_TOKENS=48
USE_CHAT_TEMPLATE=True

ROOT=Path('/content') if Path('/content').exists() else Path.cwd()
DATA_DIR=ROOT/'ctd_data'; DATA_DIR.mkdir(parents=True,exist_ok=True)
try:
    from google.colab import drive
    drive.mount('/content/drive',force_remount=False)
    DRIVE_ROOT=Path('/content/drive/MyDrive/llm-tuning-playground')
except Exception:
    DRIVE_ROOT=ROOT/'llm-tuning-playground'
RESULT_DIR=DRIVE_ROOT/'results/19'; RESULT_DIR.mkdir(parents=True,exist_ok=True)
RESULT_CSV=RESULT_DIR/'19_strong_model_results.csv'
SUMMARY_CSV=RESULT_DIR/'19_strong_model_summary.csv'
CONFIG_JSON=RESULT_DIR/'19_strong_model_config.json'

config=dict(model=MODEL_NAME,seed=SEED,splits=SPLITS,n_eval=N_EVAL,batch_size=BATCH_SIZE,max_new_tokens=MAX_NEW_TOKENS,use_chat_template=USE_CHAT_TEMPLATE,training='none; zero-shot sanity check')
CONFIG_JSON.write_text(json.dumps(config,indent=2),encoding='utf-8')
print('CUDA:',torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')
print('Results:',RESULT_DIR)


In [ ]:
# CTD acquisition and parsing; same benchmark family as Experiments 15--18.
CHEM_NAME='CTD_chem_gene_ixns.tsv.gz'
GD_NAMES=['CTD_curated_genes_diseases.tsv.gz','CTD_genes_diseases.tsv.gz']

def valid_gzip(path,min_bytes=10000):
    path=Path(path)
    if not path.exists() or path.stat().st_size<min_bytes:return False
    try:
        with open(path,'rb') as f:
            if f.read(2)!=b'\x1f\x8b':return False
        with gzip.open(path,'rb') as f:f.read(256)
        return True
    except Exception:return False

def find_local(name):
    for p in [Path.cwd()/name,ROOT/name,DATA_DIR/name,Path('/content/drive/MyDrive')/name,Path('/content/drive/MyDrive/ctd')/name,Path('/content/drive/MyDrive/data')/name]:
        if valid_gzip(p):print('Found:',p);return p
    return None

def download_ctd(name):
    dest=DATA_DIR/name
    for url in [f'https://ctdbase.org/reports/{name}',f'https://ctdbase.org/downloads/{name}',f'http://ctdbase.org/reports/{name}']:
        try:
            print('Trying:',url)
            with requests.get(url,stream=True,timeout=(20,300),allow_redirects=True,headers={'User-Agent':'Mozilla/5.0'}) as r:
                r.raise_for_status()
                with open(dest,'wb') as f:
                    for ch in r.iter_content(1024*1024):
                        if ch:f.write(ch)
            if valid_gzip(dest):return dest
        except Exception as e:print(' failed:',type(e).__name__,str(e)[:120])
        dest.unlink(missing_ok=True)
    return None

def ensure_ctd(names):
    if isinstance(names,str):names=[names]
    for n in names:
        p=find_local(n)
        if p:return p
    for n in names:
        p=download_ctd(n)
        if p:return p
    raise FileNotFoundError(names)

def read_ctd(path,expected_any):
    header=None
    with gzip.open(path,'rt',encoding='utf-8',errors='replace') as f:
        for line in f:
            if not line.startswith('#'):break
            s=line.lstrip('#').strip()
            if '\t' in s:
                cols=[x.strip() for x in s.split('\t')]
                if any(x in cols for x in expected_any):header=cols
    if header is None:raise ValueError(path)
    return pd.read_csv(path,sep='\t',comment='#',compression='gzip',dtype=str,low_memory=False,header=None,names=header)

def pick(df,names):
    for n in names:
        if n in df.columns:return n
    raise KeyError(names)

cg=read_ctd(ensure_ctd(CHEM_NAME),['ChemicalName','ChemicalID','GeneSymbol','GeneID'])
gd=read_ctd(ensure_ctd(GD_NAMES),['GeneSymbol','GeneID','DiseaseName','DiseaseID'])
c_name=pick(cg,['ChemicalName']);c_id=pick(cg,['ChemicalID']);g_sym1=pick(cg,['GeneSymbol']);g_id1=pick(cg,['GeneID'])
g_sym2=pick(gd,['GeneSymbol']);g_id2=pick(gd,['GeneID']);d_name=pick(gd,['DiseaseName']);d_id=pick(gd,['DiseaseID'])
cg2=cg[[c_name,c_id,g_sym1,g_id1]].dropna().drop_duplicates();gd2=gd[[g_sym2,g_id2,d_name,d_id]].dropna().drop_duplicates()
cg2.columns=['ChemicalName','ChemicalID','GeneSymbol','GeneID'];gd2.columns=['GeneSymbol','GeneID','DiseaseName','DiseaseID']
paths=cg2.merge(gd2,on=['GeneSymbol','GeneID'],how='inner').drop_duplicates()
paths=paths[(paths.ChemicalName.str.len()<100)&(paths.DiseaseName.str.len()<120)].reset_index(drop=True)
edge_pool=gd2[['GeneSymbol','DiseaseName']].drop_duplicates().reset_index(drop=True)
assert len(paths)>3000
print('Two-hop paths:',len(paths))


In [ ]:
# Benchmark construction and scoring.
def render_prompt(row,edges):
    lines=[f'- {g} -> {d}' for g,d in edges]
    return ('Use only the supplied evidence. Determine the disease supported by the path from the queried chemical through the queried gene. If no supplied gene-disease relation supports the queried gene, answer exactly: No supported path.\n\n'+f'Chemical: {row.ChemicalName}\nGene: {row.GeneSymbol}\nEvidence:\n'+'\n'.join(lines))

def positive_edges(row,k,rng):
    edges=[(str(row.GeneSymbol),str(row.DiseaseName))]
    pool=edge_pool[(edge_pool.GeneSymbol!=row.GeneSymbol)&(edge_pool.DiseaseName!=row.DiseaseName)]
    if k:
        sub=pool.sample(n=k,random_state=rng.randint(0,2**31-1));edges += [(str(g),str(d)) for g,d in sub.itertuples(index=False,name=None)]
    rng.shuffle(edges);return edges

def no_path_edges(row,k,rng,lexical=False):
    pool=edge_pool[(edge_pool.GeneSymbol!=row.GeneSymbol)&(edge_pool.DiseaseName!=row.DiseaseName)].copy();edges=[]
    if lexical:
        sym=str(row.GeneSymbol);near=pool[pool.GeneSymbol.astype(str).str.startswith(sym[:max(1,min(2,len(sym)))])]
        if len(near):
            x=near.sample(1,random_state=rng.randint(0,2**31-1)).iloc[0];edges.append((str(x.GeneSymbol),str(x.DiseaseName)));pool=pool[pool.GeneSymbol!=x.GeneSymbol]
    need=k-len(edges)
    if need>0:
        sub=pool.sample(need,random_state=rng.randint(0,2**31-1));edges += [(str(g),str(d)) for g,d in sub.itertuples(index=False,name=None)]
    rng.shuffle(edges);return edges

def counterfactual_edges(row,rng):
    c=edge_pool[(edge_pool.GeneSymbol==row.GeneSymbol)&(edge_pool.DiseaseName!=row.DiseaseName)]
    cf=str(c.sample(1,random_state=rng.randint(0,2**31-1)).iloc[0].DiseaseName) if len(c) else str(edge_pool[edge_pool.DiseaseName!=row.DiseaseName].sample(1,random_state=rng.randint(0,2**31-1)).iloc[0].DiseaseName)
    return [(str(row.GeneSymbol),cf)],cf

def heldout_eval(df,col,seed,n_eval):
    r=np.random.default_rng(seed);ents=df[col].dropna().unique().copy();r.shuffle(ents);cut=max(1,int(.8*len(ents)));te=set(ents[cut:])
    pool=df[df[col].isin(te)].drop_duplicates(['ChemicalID','GeneID','DiseaseID'])
    return pool.sample(n_eval,random_state=1000+seed).reset_index(drop=True)

def item(row,edges,target,typ):return {'target_disease':None if target is None else str(target),'prompt':render_prompt(row,edges),'answer_type':typ}

def make_eval_sets(df,seed):
    rng=random.Random(20000+seed);out={k:[] for k in ['clean','distractor_5','hard_no_path','lexical_no_path','counterfactual']}
    for _,row in df.iterrows():
        out['clean'].append(item(row,positive_edges(row,0,rng),row.DiseaseName,'positive'))
        out['distractor_5'].append(item(row,positive_edges(row,5,rng),row.DiseaseName,'positive'))
        out['hard_no_path'].append(item(row,no_path_edges(row,5,rng,False),None,'no_path'))
        out['lexical_no_path'].append(item(row,no_path_edges(row,5,rng,True),None,'no_path'))
        e,cf=counterfactual_edges(row,rng);out['counterfactual'].append(item(row,e,cf,'positive'))
    return out

def norm(s):return re.sub(r'\s+',' ',str(s).strip().lower())
def score_one(x,p):
    p=norm(p)
    return ('no supported path' in p) if x['answer_type']=='no_path' else (norm(x['target_disease']) in p and 'no supported path' not in p)
def score_set(items,preds):return float(np.mean([score_one(x,p) for x,p in zip(items,preds)]))


In [ ]:
# Load a single substantially stronger model in 4-bit for a standard Colab GPU.
set_seed(SEED)
tokenizer=AutoTokenizer.from_pretrained(MODEL_NAME,use_fast=True)
tokenizer.pad_token=tokenizer.pad_token or tokenizer.eos_token
tokenizer.padding_side='left'
bnb=BitsAndBytesConfig(load_in_4bit=True,bnb_4bit_quant_type='nf4',bnb_4bit_compute_dtype=torch.float16,bnb_4bit_use_double_quant=True)
model=AutoModelForCausalLM.from_pretrained(MODEL_NAME,quantization_config=bnb,device_map='auto',torch_dtype=torch.float16)
model.eval()
print('Loaded',MODEL_NAME)

def format_for_model(prompt):
    if USE_CHAT_TEMPLATE:
        messages=[{'role':'user','content':prompt+'\n\nReturn only the answer.'}]
        return tokenizer.apply_chat_template(messages,tokenize=False,add_generation_prompt=True)
    return prompt+'\nAnswer:'

@torch.inference_mode()
def generate_batch(prompts):
    texts=[format_for_model(p) for p in prompts]
    enc=tokenizer(texts,return_tensors='pt',padding=True,truncation=True,max_length=768).to(model.device)
    out=model.generate(**enc,max_new_tokens=MAX_NEW_TOKENS,do_sample=False,pad_token_id=tokenizer.pad_token_id,eos_token_id=tokenizer.eos_token_id)
    gen=out[:,enc['input_ids'].shape[1]:]
    return tokenizer.batch_decode(gen,skip_special_tokens=True)

def predict(items):
    preds=[]
    for i in range(0,len(items),BATCH_SIZE):
        batch=items[i:i+BATCH_SIZE]
        preds.extend(generate_batch([x['prompt'] for x in batch]))
    return preds


In [ ]:
# Run: 3 split definitions x 5 conditions x 50 examples = 750 generations.
rows=[]
t0=time.time()
for split in SPLITS:
    eval_df=heldout_eval(paths,split,SEED,N_EVAL)
    sets=make_eval_sets(eval_df,SEED)
    for condition,items in sets.items():
        preds=predict(items)
        acc=score_set(items,preds)
        print(f'{split:10s} {condition:16s} acc={acc:.3f}')
        rows.append(dict(model=MODEL_NAME,split=split,seed=SEED,condition=condition,n=len(items),accuracy=acc))
        pd.DataFrame(rows).to_csv(RESULT_CSV,index=False)

res=pd.DataFrame(rows)
summary=res.pivot_table(index='split',columns='condition',values='accuracy',aggfunc='mean').reset_index()
summary.to_csv(SUMMARY_CSV,index=False)
display(summary)
print('Elapsed min:',round((time.time()-t0)/60,1))


In [ ]:
# Minimal stop/go diagnostic. This is only a sanity heuristic, not a statistical decision rule.
means=res.groupby('condition').accuracy.mean()
d5=float(means.get('distractor_5',np.nan)); hard=float(means.get('hard_no_path',np.nan))
print(f'Across-split mean: D5={d5:.3f}, Hard NP={hard:.3f}')
if d5>=0.95 and hard>=0.95:
    print('CEILING-LIKE: a 7B instruction model solves both axes nearly perfectly. Reconsider broad LLM-reliability claims for this benchmark.')
elif d5>=0.90 and hard<0.80:
    print('DISSOCIATION PERSISTS: strong selection with materially weaker sufficiency remains in the 7B zero-shot model.')
else:
    print('MIXED: inspect all five conditions and per-split values before deciding whether to expand the strong-model study.')
print('Saved:',RESULT_CSV)
